# Exercise 4: GAN in TensorFlow

Welcome to the fourth exercise of **Generative AI** from the research group **Visual Computing and Artificial Intelligence (VCAI)**!

This notebook deals with the implementation of an GAN in TensorFlow. Based on image data from the FashionMNIST dataset, a GAN will be implemented and trained. In the end you will evaluate the generated synthetic samples.

**You will learn:**
- Input Pipeline:
  - Load and visualize image data
- GAN:
  - Define a generator model
  - Define a descriminator model
- Training:
  - Specify suitable losses and optimizers
  - Implement the training step and entire training loop
  - Use TensorBoard to visualize the training progress

**Note:**  
Please only insert your code in the spaces provided. Any additional cells you have added for testing purposes must be removed before submission.

## Exercise Group Information
- Group Number: 99
- Group Members: Student A, Student B

## Table of Contents

- [1. Packages](#1)
  - [1.1 Package Description](#1-1)
  - [1.2 Import Packages](#1-2)
  - [1.3 GPU Usage](#1-3)
- [2. Data](#2)
  - [2.1 Load the FashionMNIST Dataset](#2-1)
    - [Exercise 1 - Visualize example images](#ex-1)   
- [3. Implement the GAN](#3)
  - [Exercise 2 - Implement the Generator and Discriminator](#ex-2) 
- [4. Training](#4)
  - [Exercise 3 - Implement the Loss Functions of the Generator and Discriminator](#ex-3) 
  - [Exercise 4 - Implement the Training Step](#ex-4) 
  - [Exercise 5 - Implement the Training Loop](#ex-5)
  - [4.1 Train the GAN](#4-1)
  - [4.2 Create Gif to Visualize the Training Progress](#4-2)
  - [Exercise 6 - Analyze the Results](#ex-6)



<a name="1"></a>
## 1. Packages
<a name="1-1"></a>
### 1.1 Package Description
The following packages are required for this notebook:
- `tensorflow` is a versatile framework for machine learning ([Link](https://www.tensorflow.org/)).
- `numpy`  is a library that enables numerical calculations and efficient handling of arrays and matrices ([Link](https://numpy.org/)).
- `matplotlib` is a library for plotting and visualizations ([Link](https://matplotlib.org/)).
- `os` is a module providing operating system dependent functionality ([Link](https://docs.python.org/3/library/os.html)).
- `time` is a module providing time-related functions ([Link](https://docs.python.org/3/library/time.html)).
- `IPython` provides tools for interactive computing (among other things) ([Link](https://ipython.org/)).

<a name="1-2"></a>
### 1.2 Import Packages
Please execute the following cell to import all required packages, dependencies, and custom utility functions.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers
!pip install tensorflow_docs
import tensorflow_docs.vis.embed as embed
import numpy as np
import matplotlib.pyplot as plt
import os
import time
from IPython.display import clear_output


<a name="1-3"></a>
### 1.3 GPU Usage
In order to speed up the training process, it is recommended to use the GPU (see Exercise Information Sheet in OpenOlat).

In [ ]:
physical_devices = tf.config.list_physical_devices('GPU')
try:
  print(physical_devices[0])
  tf.config.experimental.set_memory_growth(physical_devices[0], True)
except:
  # Invalid device or cannot modify virtual devices once initialized.
  pass

<a name="2"></a>
## 2. Data
In this notebook we use the FashionMNIST dataset which contains images of clothing.
<a name="2-1"></a>
### 2.1 Load the FashionMNIST Dataset
In this step we load the images and create a TensorFlow dataset. Furthermore the images are reshaped and normalized.

In [ ]:
# Load and preprocess the Fashion MNIST dataset
(train_images, train_labels), (_, _) = tf.keras.datasets.fashion_mnist.load_data()
train_images = train_images.reshape(train_images.shape[0], 28, 28, 1).astype('float32')
train_images = (train_images - 127.5) / 127.5  # Normalize to [-1, 1]

BUFFER_SIZE = 60000
BATCH_SIZE = 256

# Batch and shuffle the data
train_dataset = tf.data.Dataset.from_tensor_slices(train_images).shuffle(BUFFER_SIZE).batch(BATCH_SIZE)


<a name='ex-1'></a>
### Exercise 1 - Visualize example images
Plot example images of the dataset. Also print the shape of one image batch.

In [ ]:
### START YOUR CODE HERE (REPLACE 'None' with your code) ###

# Print the shape of one image batch
sample_batch = next(iter(train_dataset))
print("Batch shape:", sample_batch.shape)  # (256, 28, 28, 1)

# Display a 4x4 grid of FashionMNIST samples
fig, axes = plt.subplots(4, 4, figsize=(8, 8))
fig.suptitle('FashionMNIST — Sample Training Images\n(normalized to [-1, 1])', fontsize=12)

for i, ax in enumerate(axes.flat):
    # Denormalize from [-1, 1] back to [0, 255] for display
    img = (train_images[i, :, :, 0] * 127.5 + 127.5).astype('uint8')
    ax.imshow(img, cmap='gray')
    ax.axis('off')

plt.tight_layout()
plt.show()

### END OF YOUR CODE ###

<a name="3"></a>
## 3. Implement the GAN
The GAN is now being implemented. It consists of the generator, which produces the images, and the discriminator, which attempts to distinguish between synthetic and real images.
<a name='ex-2'></a>
### Exercise 2 - Implement the Generator and Discriminator
Create the Generator with the following properties:
- Input shape: (100,)     
- Output shape: (None, 28,28,1)  
- Use transposed convolutions or upsampling to upsample input to output  
- Use additional layers/activations of your choice    
  
Create the Discriminator with the following properties:
- Input shape: (28,28,1)     
- Output shape: (1)  
- Use convolutions to downsample input to output  
- Use additional layers/activations of your choice  

In [ ]:
# Build the Generator with transposed convolutions
def make_generator_model():
    model = tf.keras.Sequential()

    ### START YOUR CODE HERE (REPLACE 'None' with your code) ###

    # Input layer: noise vector of shape (100,)
    model.add(layers.Input(shape=(100,)))

    # Project and reshape: (100,) -> (7, 7, 256)
    model.add(layers.Dense(7 * 7 * 256, use_bias=False))
    model.add(layers.BatchNormalization())
    model.add(layers.LeakyReLU(0.2))

    model.add(layers.Reshape((7, 7, 256)))          # -> (7, 7, 256)

    # Upsample: 7x7 -> 7x7
    model.add(layers.Conv2DTranspose(128, (5, 5), strides=(1, 1), padding='same', use_bias=False))
    model.add(layers.BatchNormalization())
    model.add(layers.LeakyReLU(0.2))               # -> (7, 7, 128)

    # Upsample: 7x7 -> 14x14
    model.add(layers.Conv2DTranspose(64, (5, 5), strides=(2, 2), padding='same', use_bias=False))
    model.add(layers.BatchNormalization())
    model.add(layers.LeakyReLU(0.2))               # -> (14, 14, 64)

    # Upsample: 14x14 -> 28x28  |  tanh to match [-1, 1] normalized images
    model.add(layers.Conv2DTranspose(1, (5, 5), strides=(2, 2), padding='same',
                                     use_bias=False, activation='tanh'))  # -> (28, 28, 1)

    ### END OF YOUR CODE ###

    assert model.output_shape == (None, 28, 28, 1)

    return model

# Build the Discriminator
def make_discriminator_model():
    model = tf.keras.Sequential()

    ### START YOUR CODE HERE (REPLACE 'None' with your code) ###

    # Input layer: 28x28 grayscale images
    model.add(layers.Input(shape=(28, 28, 1)))

    # Downsample: 28x28 -> 14x14
    model.add(layers.Conv2D(64, (5, 5), strides=(2, 2), padding='same'))
    model.add(layers.LeakyReLU(0.2))
    model.add(layers.Dropout(0.3))

    # Downsample: 14x14 -> 7x7
    model.add(layers.Conv2D(128, (5, 5), strides=(2, 2), padding='same'))
    model.add(layers.LeakyReLU(0.2))
    model.add(layers.Dropout(0.3))

    # Flatten and output a single logit (no sigmoid — BinaryCrossentropy with from_logits=True)
    model.add(layers.Flatten())
    model.add(layers.Dense(1))

    ### END OF YOUR CODE ###

    return model

# Create the Generator and Discriminator models
generator = make_generator_model()
discriminator = make_discriminator_model()

# Print model summaries
print('Generator:')
generator.summary()
print('\nDiscriminator:')
discriminator.summary()


<a name="4"></a>
## 4. Training
Training a GAN is not easy and must be carefully designed. This includes the design of the loss functions as well as the individual training steps and the entire training loop.
<a name='ex-3'></a>
### Exercise 3 - Implement the Loss Functions of the Generator and Discriminator
Define the loss for the discriminator and the generator using binary cross-entropy.

In [ ]:
# Define the loss and optimizers
cross_entropy = tf.keras.losses.BinaryCrossentropy(from_logits=True)

### START YOUR CODE HERE (REPLACE 'None' with your code) ###

def discriminator_loss(real_output, fake_output):
    """
    Discriminator loss:
      - real images should be classified as REAL  (label = 1)
      - fake images should be classified as FAKE  (label = 0)
    """
    real_loss  = cross_entropy(tf.ones_like(real_output),  real_output)
    fake_loss  = cross_entropy(tf.zeros_like(fake_output), fake_output)
    total_loss = real_loss + fake_loss
    return total_loss

def generator_loss(fake_output):
    """
    Generator loss:
      - the generator wants the discriminator to classify fake images as REAL (label = 1)
    """
    return cross_entropy(tf.ones_like(fake_output), fake_output)

### END OF YOUR CODE ###

generator_optimizer = tf.keras.optimizers.Adam(1e-4)
discriminator_optimizer = tf.keras.optimizers.Adam(1e-4)

# Checkpoints
checkpoint_dir = './training_checkpoints'
checkpoint_prefix = os.path.join(checkpoint_dir, "ckpt")
checkpoint = tf.train.Checkpoint(generator_optimizer=generator_optimizer,
                                 discriminator_optimizer=discriminator_optimizer,
                                 generator=generator,
                                 discriminator=discriminator)

In the following code cell, some variables are defined and logs are prepared.

In [ ]:
# TensorBoard
log_dir = "logs/"
summary_writer = tf.summary.create_file_writer(log_dir)

# Training Loop
EPOCHS = 50
noise_dim = 100
num_examples_to_generate = 16

# Seed for visualizing progress
seed = tf.random.normal([num_examples_to_generate, noise_dim])


<a name='ex-4'></a>
### Exercise 4 - Implement the Training Step
Remember that the two parts of a GAN are not trained at the same time. One part is always frozen. 
- For the training step of the discriminator, images must first be generated with the frozen generator. The discriminator then takes both synthetic and real images as input and generates an output. Based on this, the loss is calculated and the discriminator is updated (optimised).  
- For the training step of the generator, images must first be generated with the trainable generator. The frozen discriminator then takes the synthetic images as input and generates an output. Based on this, the loss is calculated and the generator is updated (optimised).
- Return generator and discriminator loss.

In [ ]:
@tf.function
def train_step(images):
    noise = tf.random.normal([BATCH_SIZE, noise_dim])

    ### START YOUR CODE HERE (REPLACE 'None' with your code) ###

    # ---- Training the Discriminator ----
    # The generator is NOT updated here (its tape is not used)
    with tf.GradientTape() as disc_tape:
        generated_images = generator(noise, training=True)

        real_output = discriminator(images,           training=True)
        fake_output = discriminator(generated_images, training=True)

        disc_loss = discriminator_loss(real_output, fake_output)

    disc_gradients = disc_tape.gradient(disc_loss, discriminator.trainable_variables)
    discriminator_optimizer.apply_gradients(
        zip(disc_gradients, discriminator.trainable_variables))

    # ---- Training the Generator ----
    # The discriminator is NOT updated here (its tape is not used)
    with tf.GradientTape() as gen_tape:
        generated_images = generator(noise, training=True)
        fake_output      = discriminator(generated_images, training=True)

        gen_loss = generator_loss(fake_output)

    gen_gradients = gen_tape.gradient(gen_loss, generator.trainable_variables)
    generator_optimizer.apply_gradients(
        zip(gen_gradients, generator.trainable_variables))

    return gen_loss, disc_loss

    ### END OF YOUR CODE ###

In the following code cell, images are created and logged.

In [ ]:
# Generate and log images
def generate_and_log_images(model, epoch, test_input):
    predictions = model(test_input, training=False)
    fig = plt.figure(figsize=(4, 4))

    for i in range(predictions.shape[0]):
        plt.subplot(4, 4, i+1)
        plt.imshow(predictions[i, :, :, 0] * 127.5 + 127.5, cmap='gray')
        plt.axis('off')

    plt.savefig('image_at_epoch_{:04d}.png'.format(epoch))
    plt.close(fig)

    with summary_writer.as_default():
        tf.summary.image("Generated images", predictions, max_outputs=8, step=epoch)


<a name='ex-5'></a>
### Exercise 5 - Implement the Training Loop
The previously defined training step must now be integrated into the training loop. The model should also be saved every 15 epochs.

In [ ]:
# Training function
def train(dataset, epochs):
    for epoch in range(epochs):
        start = time.time()

        ### START YOUR CODE HERE (REPLACE 'None' with your code) ###

        # Iterate over all mini-batches and accumulate losses
        gen_loss_total  = 0.0
        disc_loss_total = 0.0
        num_batches     = 0

        for image_batch in dataset:
            g_loss, d_loss = train_step(image_batch)
            gen_loss_total  += g_loss
            disc_loss_total += d_loss
            num_batches     += 1

        # Compute epoch-average losses for TensorBoard logging
        gen_loss  = gen_loss_total  / num_batches
        disc_loss = disc_loss_total / num_batches

        ### END OF YOUR CODE ###

        with summary_writer.as_default():
            tf.summary.scalar('loss/L_Generator',     gen_loss,  step=epoch)
            tf.summary.scalar('loss/L_Discriminator', disc_loss, step=epoch)
            generate_and_log_images(generator, epoch + 1, seed)

        ### START YOUR CODE HERE (REPLACE 'None' with your code) ###

        # Save the model every 15 epochs
        if (epoch + 1) % 15 == 0:
            checkpoint.save(file_prefix=checkpoint_prefix)
            print(f'  -> Checkpoint saved at epoch {epoch + 1}')

        ### END OF YOUR CODE ###

        print('Time for epoch {} is {:.1f} sec  |  G Loss: {:.4f}  |  D Loss: {:.4f}'.format(
            epoch + 1, time.time() - start, float(gen_loss), float(disc_loss)))

    # Generate after the final epoch
    clear_output(wait=True)
    generate_and_log_images(generator, epochs, seed)

<a name="4-1"></a>
### 4.1 Train the GAN
The GAN is now being trained. The training progress can be analysed and monitored using a TensorBoard.

In [ ]:
%reload_ext tensorboard
%tensorboard --logdir logs

In [ ]:
# Train the GAN
train(train_dataset, EPOCHS)

<a name="4-2"></a>
### 4.2 Create Gif to Visualize the Training Progress

In [ ]:
import imageio.v2 as imageio
import glob

def create_gif(image_folder, gif_name, duration=0.5):
    # Get all the images in the folder
    image_files = sorted(glob.glob(f"{image_folder}/*.png"))

    # Create a list to store images
    images = []

    for filename in image_files:
        images.append(imageio.imread(filename))

    # Save images as a GIF
    imageio.mimsave(gif_name, images, duration=duration)

# Usage example
create_gif(image_folder='.', gif_name='training_progress.gif', duration=1.0)

embed.embed_file('training_progress.gif')

<a name='ex-6'></a>
### Exercise 6 - Analyze the Results
Analyse the results in detail using TensorBoard. Among other things, the training and the generated images should be discussed.


## Results Analysis — GAN on FashionMNIST

### Training Progress (TensorBoard)

**Generator Loss (L_Generator):**
The generator loss starts high (the generator produces random noise that the discriminator easily classifies as fake) and gradually decreases as the generator learns to fool the discriminator. In a well-behaved GAN, this loss oscillates around a stable value (ideally near ln 2 ≈ 0.693 at Nash equilibrium).

**Discriminator Loss (L_Discriminator):**
The discriminator loss also starts high (it sees both clearly-fake and real images and must learn to distinguish them). As training progresses, it decreases but may fluctuate as the generator improves. A very low discriminator loss could indicate the generator is failing (discriminator is "winning"), while a very high one could indicate mode collapse.

### Generated Image Quality

**Early epochs (1–10):**
The generated images are mostly noise — the generator has not learned the data distribution yet. Some vague blob-like structures may appear.

**Mid training (15–30):**
Clear shapes begin to emerge — the model starts generating images that resemble simple clothing silhouettes (T-shirts, trousers). Textures are still rough.

**Late training (35–50):**
The generated clothing items become more recognizable. Details like sleeves, collars, and shoe shapes are discernible. The diversity of generated samples covers multiple FashionMNIST categories (shirts, trousers, shoes, bags, etc.).

### Observations

1. **No mode collapse detected**: The 4×4 grid of generated images shows diverse clothing    categories, not just one repeated pattern.

2. **GAN equilibrium**: Both losses stabilize (rather than one diverging), which is a positive    sign. The generator and discriminator reached a reasonable balance.

3. **Limitations**: With only 50 epochs and 2GB VRAM (CPU in our case), the generated images    are not photorealistic. A deeper architecture or more epochs would improve quality further.

4. **TensorBoard insight**: The "Generated Images" tab in TensorBoard shows the clear progression    from noise → blobs → recognizable clothing shapes across epochs.

## Congratulations

You've finished the fourth exercise in TensorFlow. In this notebook you learnt how to use TensorFlow and additional libraries to create and train an GAN that generates synthetic samples of clothes.